# O pipeline em produção

## Task 4: colocar o pipeline em produção, rodando fora do notebook, e carregar a Copa de 2026

O notebook é para explorar. Em produção, o job **roda sozinho**, chamado por um agendador, e é **versionado** como
código.

In [ ]:
import sys
sys.path.insert(0, "../../src")

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession, functions as F

construtor = (
    SparkSession.builder
    .appName("arena-pipeline")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(construtor).getOrCreate()

RAW = "../../data/raw"
RAW_2026 = "../../data/raw_2026"
OUT = "../../data/out"

spark

A célula abaixo já vem pronta: o mesmo código do vídeo 5.1 (`ler_partidas` e o histórico).

In [ ]:
# vídeo 5.1: as partidas do projeto
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas

def ler_partidas(pasta):
    return limpar_partidas(
        spark.read.option("header", "true").schema(MATCHES).csv(f"{pasta}/WorldCupMatches.csv")
    )

historico = ler_partidas(RAW)

## O script do pipeline

1. **ler** o CSV de partidas; 2. **transformar** com as funções da aula 2; 3. **gravar** em Delta por ano, com
overwrite dinâmico. As pastas chegam como parâmetros (`--raw` e `--out`).

- **Sem `master`**: quem executa decide onde o job roda.
- **As configurações do Delta** (5.1) na sessão; o pacote vem pelo comando que executa o script.
- **`argparse`** lê os parâmetros da linha de comando.
- O resto já foi escrito: `ler_partidas` (5.1, com a sessão como parâmetro), as funções da aula 2 e a gravação dos vídeos 5.2 e 5.3.

## Executando com spark-submit

`spark-submit` envia um script para o Spark executar. No terminal do Jupyter, em `~/work`:

```bash
spark-submit --master "local[*]" --packages io.delta:delta-spark_4.2_2.13:4.4.0 \
  src/wc_pipeline/pipeline.py --raw data/raw --out data/out/producao
```

`--packages` traz o pacote do Delta, antes de o script começar.

A próxima célula já vem pronta: ela só apaga a saída de produção anterior.

In [ ]:
import shutil
shutil.rmtree(f"{OUT}/producao", ignore_errors=True)

## A Copa de 2026

Antes de carregar, a auditoria de nulos do vídeo 2.2:

Quantas partidas chegaram sem público? Um campo vazio vira `null` na leitura, e o pipeline não quebra. As agregações
ignoram os nulos (aula 2).

A carga, **duas vezes**:

Compare o total das duas execuções: um pipeline **idempotente** dá o mesmo resultado ao rodar de novo.

## Conferindo a saída

Como no vídeo 2.3, `assert` para conferir, agora lendo a saída do pipeline:

## Onde gravar em produção

| Lugar | Exemplo de caminho |
|---|---|
| disco local | `data/out/producao` |
| armazenamento na nuvem | `s3a://bucket/caminho` (AWS), `gs://` (Google), `abfss://` (Azure) |
| catálogo | `saveAsTable("banco.tabela")` (vídeo 3.4) |

**Nunca escreva chaves de acesso no código.**

## Do notebook para o dia a dia da empresa

| No curso | No dia a dia da empresa |
|---|---|
| `spark-submit` numa célula | um agendador (como o Airflow) roda o mesmo comando todo dia |
| `--master "local[*]"` | um cluster (`yarn`, `k8s://...`), Databricks ou EMR; o script não muda |
| `data/raw` e `data/out` | um bucket na nuvem (`s3a://`, `gs://`, `abfss://`) |
| uma Copa nova a cada 4 anos | um lote novo todo dia, regravando só a sua pasta (overwrite dinâmico) |
| conferência com `assert` | a conferência dentro do job: se falhar, o time é avisado antes do painel |
| leitura no notebook | o painel lê só a Copa que precisa; o time de analytics consulta em SQL |

Num cluster, o shuffle atravessa a rede de verdade: a aula 4 vale ainda mais. **Idempotência** e **código
versionado** são o que permite rodar isso todo dia sem susto.

## Demanda da aula 5 atendida

| Etapa | Entrega |
|---|---|
| Task 1 | `overwrite` em Delta Lake |
| Task 2 | saída por Copa, com partition pruning |
| Task 3 | um arquivo por pasta e overwrite dinâmico |
| Task 4 | `pipeline.py` com `spark-submit`, 2026 carregada sem duplicar, saída conferida |

## O projeto completo

| Aula | Entrega |
|---|---|
| 1 | ambiente no ar e arquivos lidos com o schema do projeto |
| 2 | limpeza e métricas em `src/wc_pipeline/transformacoes.py` |
| 3 | views, consulta em `sql/`, rankings e uma tabela no catálogo |
| 4 | os passes por partida e por Copa em Parquet, e o job lido no plano e na Spark UI |
| 5 | `src/wc_pipeline/pipeline.py`, Delta Lake por Copa e a Copa de 2026 |

Próximo passo: a atividade "Subindo o projeto no Git".

## Para consultar depois

- **Opções do `spark-submit`:** `--conf chave=valor` muda uma configuração só naquela execução, `--packages` traz
  pacotes como o do Delta, e `spark-submit --help` lista as outras.
- **Agendadores:** o Airflow, por exemplo, tem um operador que chama o `spark-submit` com os parâmetros de cada dia.
- **Conferências maiores:** em projetos grandes, as conferências com `assert` viram uma suíte de testes automatizados,
  rodada antes de cada mudança no código.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()